# TinyLLM Workshop：兩小時，練一個 LLM、塞進 ESP32

今天的完整旅程：

```
資料 → 練 tokenizer(BPE) → 訓練模型 → 量化(Q8_0) → 轉成 C header → 燒進 ESP32 → 看它寫故事
```

**開始前**：`Runtime > Change runtime type` 選 **T4 GPU**（沒有 GPU 也能跑，訓練會慢一些）。

跟著講師一格一格跑。每格左上角的 ▶ 按下去就對了。

In [ ]:
# 環境進備：抓課程 repo、編譯工具（約 1 分鐘）
import os
REPO_URL = "https://github.com/Ricky610329/tiny_llm_edge_deploied.git"
if not os.path.exists('course'):
    !git clone -q {REPO_URL} course
%cd course
!bash tools/build.sh

In [ ]:
%%bash
# 下載一顆別人訓練好的模型（等下當示範，也是今天的保底備案）
mkdir -p models
base=https://huggingface.co/karpathy/tinyllamas/resolve/main/stories260K
for f in stories260K.bin stories260K.pt; do
  [ -f models/$f ] || wget -q $base/$f -O models/$f
done
ls -l models

## 1. 先看終點：一個 1MB 的語言模型長什麼樣

這顆 26 萬參數的模型（GPT-4 的千萬分之一），今天課程結束時，你會做出一顆自己的、跑在一塊幾十塊錢的晶片上。

In [ ]:
!./bin/run models/stories260K.bin -z models/tok512.bin -t 0.8 -n 200 -i "Once upon a time"

## 2. Tokenizer：文字怎麼變成數字

模型不認識文字，只認識編號。**Tokenizer 就是「文字 ↔ 編號」的翻譯機**，而它是從資料裡「學」出來的：

- **資料**：TinyStories——幾百萬篇給幼兒的英文短故事（今天只抓 2 個分片，約 280MB）。
- **演算法**：BPE（Byte Pair Encoding）。從單一字元出發，**反覆把「最常相鄰出現的一對」合併成新 token**，直到詞彙表達到目標大小。合併準則就這一條：頻率最高者先合併。
- **詞彙表大小**：我們用 512——小到晶片端好處理，大到常見英文詞綴都能學會。

下面兩格：抓資料、**親手訓練一顆 tokenizer**（注意看 log：它會依序學會 `th`、`ing`、`▁the` 這些合併）。

In [ ]:
!python tools/fetch_shards.py --n 2

In [ ]:
!cd train/llama2.c && echo n | python tinystories.py train_vocab --vocab_size=512 2>&1 | grep -E "Added|Saving|Size" | head -40

In [ ]:
# 玩一下你剛練出來的 tokenizer
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file='train/llama2.c/data/tok512.model')
s = "Once upon a time, there was a little robot."
print('pieces:', sp.encode(s, out_type=str))
print('ids:   ', sp.encode(s))
print('vocab 大小:', sp.get_piece_size())
# 觀察：常見的字（time、was）是一整塊；罕見的字（robot）被拆成好幾塊

In [ ]:
# 把 tokenizer 轉成 C 程式吃的格式，並把 2 個分片的故事全部預先切成 token（約 2-4 分鐘）
!cd train/llama2.c && python tokenizer.py -t data/tok512.model
!cd train/llama2.c && python tinystories.py pretokenize --vocab_size=512 2>&1 | tail -3

## 3. 模型：一張圖 + 一鍵訓練

今天不進數學，只記住這張圖（細節在 `docs/advanced/` 給想深入的人）：

```
token 編號 → 查表變向量(64 維)
              │
              ▼
    ┌──────────────────────┐
    │  注意力：回頭看前文     │  ←「這句的主角是誰?」
    │  前饋層：查內建知識     │  ←「這種句型接什麼詞?」
    └──────────────────────┘ × 5 層
              │
              ▼
    512 個分數 = 下一個 token 是誰的機率
```

我們的配置：64 維、5 層、約 28 萬參數。**訓練目標只有一句話：猜對下一個 token**（第 2 節切好的資料就是無限量的「猜謎題庫」）。

下一格發射訓練，**GPU 上約 3-5 分鐘**——跑的時候聽講師講第 4 節。

In [ ]:
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dt = 'float16' if dev == 'cuda' else 'float32'
iters = 3000 if dev == 'cuda' else 800   # CPU 備援：少練一點，故事會比較粗糙
print(f'training on {dev}, {iters} iters')
!cd train/llama2.c && python train.py \
  --vocab_source=custom --vocab_size=512 \
  --dim=64 --n_layers=5 --n_heads=8 --n_kv_heads=4 --multiple_of=64 \
  --max_seq_len=256 --batch_size=32 \
  --max_iters={iters} --eval_interval=500 --eval_iters=10 --warmup_iters=100 \
  --device={dev} --dtype={dt} --compile=False --out_dir=out_workshop 2>&1 | grep -vE "^\\d+ \\| loss" | tail -20

## 4. 量化：把 4 bytes 壓成 1 byte（訓練跑的時候聽這段）

訓練出來的權重是 fp32——每個參數 4 bytes。ESP32 的問題不只是塞不塞得下，而是**每產生一個字都要把全部權重從 flash 讀一遍**——權重多大，速度就多慢。

我們用 **Q8_0 量化**，規則簡單到可以口述：

> 權重每 **64 個一組**。每組找出絕對值最大的，`scale = max/127`，
> 然後組內每個數除以 scale、四捨五入成 int8（±127）。
> 存檔 = 64 個 int8 + 一個 fp32 的 scale。

- 成本：**每參數 1.06 bytes**（原本 4）→ 模型直接小 3.8 倍、快 3.8 倍
- 品質代價：實測 BPB 0.8132 → 0.8135，**幾乎零損失**（訓練好的權重分布集中，8 bits 綽綽有餘）
- 為什麼分組？防離群值——一個特大的權重只會拖累同組 64 個，不會毀掉整個矩陣

In [ ]:
# 量化你剛訓練的模型
!python tools/quantize.py train/llama2.c/out_workshop/ckpt.pt models/mymodel_q80.bin 2>&1 | tail -2

In [ ]:
# 你的模型會寫什麼故事？（用你自己練的 tokenizer！）
!./bin/runq models/mymodel_q80.bin -z train/llama2.c/data/tok512.bin -t 0.8 -n 200 -i "Once upon a time"

In [ ]:
# 幫你的模型打分數：bits-per-byte（越低越好；別人練很久的 baseline 是 0.8135）
!./bin/eval_bpb_q models/mymodel_q80.bin -z train/llama2.c/data/tok512.bin -f eval/validation_100.txt -w 128

## 5. 上板：把模型變成一個 C 陣列

最後一步的魔法很單純：**把量化模型轉成一個巨大的 C 常數陣列**，跟程式一起編譯。
ESP32 上 `const` 資料放在 flash、自動映射成可直接讀的記憶體——所以權重一個 byte 都不佔 RAM。

下一格會產生 `model_data.h` 並下載到你電腦——接下來的燒錄步驟看講義（Arduino IDE 開 `firmware/tinyllm_arduino/`，設定照表，Upload）。

In [ ]:
!python tools/export_header.py models/mymodel_q80.bin train/llama2.c/data/tok512.bin
try:
    from google.colab import files
    files.download('firmware/tinyllm_arduino/model_data.h')
except ImportError:
    print('（非 Colab 環境：檔案在 firmware/tinyllm_arduino/model_data.h）')

### 備案：如果前面訓練或 tokenizer 出了問題

跑下面這格，改用預訓練模型 + 官方 tokenizer 完成燒錄環節（其他步驟完全一樣）：

In [ ]:
# （備案）用 stories260K 產 header
!python tools/quantize.py models/stories260K.pt models/stories260K_q80.bin 2>&1 | tail -1
!python tools/export_header.py models/stories260K_q80.bin models/tok512.bin
try:
    from google.colab import files
    files.download('firmware/tinyllm_arduino/model_data.h')
except ImportError:
    pass

## 6. 結語：你今天拿到的是「能跑就好」的版本

板上的推論引擎是**故意寫慢的**：單核心（第二顆核心整顆閒著）、純 scalar 運算、flash 只跑半速模式。
即使這樣，它也有 ~20 tok/s——超過人類朗讀速度。

**還能快多少？保守估計 4 倍以上。** 方向給你，code 自己寫：

| 方向 | 提示 |
|---|---|
| Flash 模式 | Arduino Tools 選單裡藏著一個兩倍頻寬的選項 |
| 雙核心 | 矩陣乘法的每一列彼此獨立 |
| 更長的記憶 | KV cache 換成 fp16，RAM 省一半 |
| 訓練更久／更大 | 你今天只練了 3000 步…… |

想深入的人：`docs/advanced/` 有完整深度版課程（架構數學、記憶體預算推導、量化陷阱、競賽規則）。
歡迎挑戰兩個紀錄：**速度 19.67 tok/s、品質 BPB 0.8135**。